# Classification

**Goal:**
In this notebook, we will perform the first step of the Responsible Data Science project. 
We will:
1. Load and preprocess the `adult.data` dataset.
2. Binarize the `Age` attribute (convert it into two groups).
3. Split the data into **Train**, **Validation**, and **Test** sets.
4. Train a classifier to predict if a person earns `>50K`.
5. Measure the performance of the model.

## Imports

In [44]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from IPython.display import display 
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Set a random state for reproducibility
RANDOM_SEED = 42

## Load the Dataset

The `adult.data` file does not contain headers (column names). We must define them manually based on the dataset documentation. 
We also need to handle spaces in the data (e.g., " White" instead of "White").

In [45]:
import os

# Define column names based on the UCI repository
columns = [
    "Age", "Workclass", "fnlwgt", "Education", "Education-Num",
    "Marital-Status", "Occupation", "Relationship", "Race", "Sex",
    "Capital-Gain", "Capital-Loss", "Hours-per-week", "Country", "Income"
]

# Define the relative path to the data file
# "../" moves up from 'notebooks' to the project root
file_path = '../data/raw/adult.data'

# Check if file exists before loading (good practice)
if not os.path.exists(file_path):
    print(f"Error: File not found at {file_path}")
else:
    # Load the data
    df = pd.read_csv(
        file_path, 
        names=columns, 
        sep=',', 
        skipinitialspace=True, 
        na_values='?'
    )

    # Display the first few rows
    print(f"Dataset shape: {df.shape}")
    display(df.head())

Dataset shape: (32561, 15)


,Age,Workclass,fnlwgt,Education,Education-Num,Marital-Status,Occupation,Relationship,Race,Sex,Capital-Gain,Capital-Loss,Hours-per-week,Country,Income
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K


## Preprocessing & Imputation Strategy

Instead of simply dropping rows with missing values (which would result in losing ~7% of our data and potentially introducing bias against certain groups), we adopt an **Imputation Strategy**.

* **Missing Values:** We replace missing entries (originally `?`) with the category `"Unknown"`. This allows the model to learn patterns associated with missing information (e.g., perhaps people with specific jobs are less likely to report them) and preserves the dataset size.
* **Age Binarization:** As per the instructions, we convert the continuous `Age` variable into a binary group based on the median:
    * `0`: Young (<= Median)
    * `1`: Senior (> Median)

In [46]:
# 1. Handling Missing Values (Imputation Strategy)

# Check missing values
print("Missing values before imputation:")
print(df.isnull().sum())

# Fill categorical missing values (Workclass, Occupation, Country) with 'Unknown'
# Note: Since we loaded with na_values='?', these are currently NaN
df = df.fillna('Unknown')

print(f"\nRemaining rows: {df.shape[0]} (We kept everyone!)")

# 2. Binarize Age
median_age = df['Age'].median()
df['Age'] = (df['Age'] > median_age).astype(int)

print("\nAge distribution (0 = Young, 1 = Old):")
print(df['Age'].value_counts())

Missing values before imputation:
Age                  0
Workclass         1836
fnlwgt               0
Education            0
Education-Num        0
Marital-Status       0
Occupation        1843
Relationship         0
Race                 0
Sex                  0
Capital-Gain         0
Capital-Loss         0
Hours-per-week       0
Country            583
Income               0
dtype: int64

Remaining rows: 32561 (We kept everyone!)

Age distribution (0 = Young, 1 = Old):
Age
0    16681
1    15880
Name: count, dtype: int64


## Encoding Data

Machine Learning models require numerical input. We cannot give them text directly.

1.  **Target Variable (`Income`):** We map `<=50K` to `0` and `>50K` to `1`.
2.  **Features:** We use **Label Encoding**. Since we filled missing values with `"Unknown"`, this category is simply assigned a unique number just like any other profession or country. This method preserves all the information for the XGBoost classifier without expanding the dataset dimensions unnecessarily.

In [47]:
from sklearn.preprocessing import LabelEncoder

# 1. Encode Target (Income)
df['Income'] = df['Income'].apply(lambda x: 1 if x == '>50K' else 0)

# 2. Separate Features (X) and Target (y)
X = df.drop('Income', axis=1)
y = df['Income']

# 3. Label Encoding
# Standard XGBoost handles numbers better. We encode strings to numbers.
categorical_columns = X.select_dtypes(include=['object']).columns

for col in categorical_columns:
    le = LabelEncoder()
    # "Unknown" will just become a number here, which is perfect
    X[col] = le.fit_transform(X[col].astype(str))

print("Data encoding complete.")
print(f"Features shape: {X.shape}")

Data encoding complete.
Features shape: (32561, 14)


## Split Data

We need three sets:
1.  **Train:** To train the model.
2.  **Validation:** To tune the model (optional, but good practice).
3.  **Test:** To evaluate the final performance.

We will split the data into 60% Train, 20% Validation, and 20% Test.

In [48]:
# First split: 80% for Train+Val, 20% for Test
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_SEED
)

# Second split: Split the 80% (X_temp) into Train (75% of temp) and Val (25% of temp)
# This results in roughly 60% Train, 20% Val, 20% Test of the original total
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.25, random_state=RANDOM_SEED
)

print(f"Training set size: {X_train.shape[0]}")
print(f"Validation set size: {X_val.shape[0]}")
print(f"Test set size: {X_test.shape[0]}")

Training set size: 19536
Validation set size: 6512
Test set size: 6513


## 5. Train the Classifier

### Model Selection Strategy
We selected **XGBoost** (eXtreme Gradient Boosting) for this project. We chose this model based on the trade-off between **performance** and **interpretability**:
1.  **High Performance:** XGBoost captures complex, non-linear relationships in the data better than linear models (like Logistic Regression). This allows us to maximize accuracy.
2.  **Explainability Readiness:** Unlike complex ensembles (like a VotingClassifier), XGBoost is fully compatible with explainability tools like **SHAP** and **Omnixai** (which are required for the "Explainability" step of this project).

In [49]:
# Initialize the XGBoost Classifier
clf = XGBClassifier(
    n_estimators=100, 
    max_depth=6, 
    learning_rate=0.1,
    random_state=RANDOM_SEED, 
    eval_metric='logloss'
)

# Train the model
clf.fit(X_train, y_train)

print("Model training complete (XGBoost).")

# AFFICHER LE MODÈLE (C'est cette ligne qui fait apparaître le "truc")
display(clf)

Model training complete (XGBoost).


,objective,'binary:logistic'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,'logloss'


### Model Configuration

We explicitly define the **hyperparameters** of our XGBoost model to ensure reproducibility and transparency. Here is what the configuration object represents:

* **`XGBClassifier`**: This is the "Identity Card" of our model. It shows all the settings used to train it.
* **`n_estimators=100`**: The number of "experts" (decision trees) we use. We use 100 trees working together to vote on the final prediction.
* **`max_depth=6`**: The complexity of each tree. A depth of 6 allows the model to learn specific patterns without memorizing the data (overfitting).
* **`learning_rate=0.1`**: The step size at each iteration. A smaller value (like 0.1) makes the training more robust and precise.
* **`random_state=42`**: Ensures that we get exactly the same results every time we run this code.

## Measure Performance

Finally, we evaluate the classifier on the **Test set**.
We look at:
* **Accuracy:** The percentage of correct predictions.
* **Classification Report:** Precision, Recall, and F1-score for each class.

In [50]:
# Predict on the test set (Using X_test, not X_test_scaled)
y_pred = clf.predict(X_test)

# Calculate Accuracy
acc = accuracy_score(y_test, y_pred)
print(f"Test Set Accuracy: {acc:.4f}\n")

# Detailed Report
print("Classification Report:")
print(classification_report(y_test, y_pred))

# Confusion Matrix
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Test Set Accuracy: 0.8781

Classification Report:
              precision    recall  f1-score   support

           0       0.90      0.94      0.92      4942
           1       0.79      0.68      0.73      1571

    accuracy                           0.88      6513
   macro avg       0.84      0.81      0.83      6513
weighted avg       0.87      0.88      0.87      6513

Confusion Matrix:
[[4652  290]
 [ 504 1067]]
